In [2]:
import mlflow
# Step 1: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://184.72.71.39:5000/")

c:\Users\Abhi\Documents\youtube-comment-analysis\mlops-youtube-comment-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Set or create an experiment
mlflow.set_experiment("ML Algos with HP Tuning")

<Experiment: artifact_location='s3://comment-analysis-bucket-994/6', creation_time=1788963985769, effective_trace_archival_retention=None, experiment_id='6', last_update_time=1788963985769, lifecycle_stage='active', name='ML Algos with HP Tuning', tags={}, trace_location=None, workspace='default'>

In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna

In [5]:
df = pd.read_csv('../data/processed/processed_comments.csv').dropna()
df.shape

(36662, 2)

In [6]:
# Remove missing values
df = df.dropna(subset=['category', 'clean_comment'])

ngram_range = (1, 3)
max_features = 1000

# Final train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# Validation split for Optuna
X_train_inner, X_val, y_train_inner, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# TF-IDF on training data only
vectorizer = TfidfVectorizer(
    ngram_range=ngram_range,
    max_features=max_features
)

X_train_inner_vec = vectorizer.fit_transform(X_train_inner)
X_val_vec = vectorizer.transform(X_val)

# SMOTE only on training data
smote = SMOTE(random_state=42)

X_train_inner_vec, y_train_inner = smote.fit_resample(
    X_train_inner_vec,
    y_train_inner
)


# Optuna objective for MultinomialNB
def objective_mnb(trial):

    alpha = trial.suggest_float(
        'alpha',
        1e-4,
        1.0,
        log=True
    )

    model = MultinomialNB(alpha=alpha)

    model.fit(X_train_inner_vec, y_train_inner)

    y_pred = model.predict(X_val_vec)

    return accuracy_score(y_val, y_pred)


# Run Optuna
study = optuna.create_study(direction="maximize")

study.optimize(
    objective_mnb,
    n_trials=30
)

best_params = study.best_params

print("Best parameters:", best_params)
print("Best validation accuracy:", study.best_value)


# Train final model on full training data
final_vectorizer = TfidfVectorizer(
    ngram_range=ngram_range,
    max_features=max_features
)

X_train_vec = final_vectorizer.fit_transform(X_train)
X_test_vec = final_vectorizer.transform(X_test)

smote = SMOTE(random_state=42)

X_train_vec, y_train = smote.fit_resample(
    X_train_vec,
    y_train
)

best_model = MultinomialNB(
    alpha=best_params['alpha']
)

best_model.fit(X_train_vec, y_train)

y_pred = best_model.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred)

print("Final accuracy:", accuracy)


# Log in MLflow
with mlflow.start_run():

    mlflow.set_tag(
        "mlflow.runName",
        "MultinomialNB_SMOTE_TFIDF_Trigrams"
    )

    mlflow.set_tag(
        "experiment_type",
        "algorithm_comparison"
    )

    mlflow.log_param("algo_name", "MultinomialNB")
    mlflow.log_param("ngram_range", str(ngram_range))
    mlflow.log_param("max_features", max_features)
    mlflow.log_param("n_trials", 30)

    mlflow.log_params(best_params)

    mlflow.log_metric("accuracy", accuracy)

    classification_rep = classification_report(
        y_test,
        y_pred,
        output_dict=True
    )

    for label, metrics in classification_rep.items():
        if isinstance(metrics, dict):
            for metric, value in metrics.items():
                mlflow.log_metric(
                    f"{label}_{metric}",
                    value
                )

    mlflow.sklearn.log_model(
        best_model,
        name="MultinomialNB_model"
    )

[I 2026-09-09 18:32:16,707] A new study created in memory with name: no-name-3e36d3e5-4b33-48fb-9957-bc1304ff478c
[I 2026-09-09 18:32:16,740] Trial 0 finished with value: 0.660927378111149 and parameters: {'alpha': 0.0002616245264207612}. Best is trial 0 with value: 0.660927378111149.
[I 2026-09-09 18:32:16,748] Trial 1 finished with value: 0.6605864302761677 and parameters: {'alpha': 0.005823491733690214}. Best is trial 0 with value: 0.660927378111149.
[I 2026-09-09 18:32:16,759] Trial 2 finished with value: 0.660927378111149 and parameters: {'alpha': 0.00021751840567393006}. Best is trial 0 with value: 0.660927378111149.
[I 2026-09-09 18:32:16,772] Trial 3 finished with value: 0.6605864302761677 and parameters: {'alpha': 0.001684173130147585}. Best is trial 0 with value: 0.660927378111149.
[I 2026-09-09 18:32:16,783] Trial 4 finished with value: 0.6607569041936584 and parameters: {'alpha': 0.0002773670903919614}. Best is trial 0 with value: 0.660927378111149.
[I 2026-09-09 18:32:16,7

Best parameters: {'alpha': 0.0002616245264207612}
Best validation accuracy: 0.660927378111149
Final accuracy: 0.6604391108686758
🏃 View run MultinomialNB_SMOTE_TFIDF_Trigrams at: http://184.72.71.39:5000/#/experiments/6/runs/319cd12ff33c4c508e41e831dfdd0186
🧪 View experiment at: http://184.72.71.39:5000/#/experiments/6
